# Economic Results Validation

1) 300W PV-only system comparison with Homer

In [10]:
import pandas as pd
import matplotlib.pyplot as plt  # for visualization
import matplotlib.ticker as mticker 
from matplotlib.ticker import MaxNLocator
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import math
import datetime
import numpy as np
from statistics import median
from statistics import mean
import seaborn as sns
import sys
sys.path.append('..')



## 1) PV only comparison

In [53]:
#variables - MAKE SURE THESE ARE IDENTICAL TO WHAT WAS USED IN THE MODEL
myNetworks = ['bowling green','borden','sunnyside','grasslands','fresh kills','park slope', 'fordham', 'central bronx']

directory = ''#final data/sept11/'

# plug-in only - no power station - file includes all sensitivity combinations
sf = 'nobattery_20260921_232701'
sd = ''#solar_only_with_azimuth_and_baseline/'

s_networkFile = f'{sd}network_df_{sf}'
s_resultsFile = f'{sd}results_df_{sf}'

In [54]:
#these are the columns with lists in them
strToList = ['dailyPVWhAC_4M','annualPVkWhDC_degraded','annualPVkWhAC_degraded','csrpMaxFlex',
             'csrpLoad1Income', 'csrpLoad2Income', 'csrpLoad1RIncome','dlrpMaxFlex',
             'csrpLoad1_flex','csrpLoad2_flex','csrpLoad1replacement_flex',
             'dlrpLoad1_flex','dlrpLoad2_flex','dlrpLoad1replacement_flex',
             'dlrpLoad1Income','dlrpLoad2Income', 'dlrpLoad1RIncome', 'annDRIncomeLoad1',
             'annDRIncomeLoad2', 'annDRIncomeLoad1R']

#these are the columns with dictionaries in them
strToDict =[ 'gridValue_load1','gridValue_load2','gridValue_load1R',
             'load1_PBP_capexSensitivity', 'load2_PBP_capexSensitivity', 'load1R_PBP_capexSensitivity',
             'load1_NPV_capexSensitivity','load2_NPV_capexSensitivity', 'load1R_NPV_capexSensitivity',
             'load1_IRR_capexSensitivity', 'load2_IRR_capexSensitivity', 'load1R_IRR_capexSensitivity',
             'load1_LCOSS_capexSensitivity', 'load2_LCOSS_capexSensitivity', 'load1R_LCOSS_capexSensitivity']

In [55]:
s_results_df = pd.read_csv(f'../results/{directory}{s_resultsFile}.csv')

# batWh_80p is the full amount, not 80%, but batWhAC_80p is correct
try:
    s_results_df['batWh']= s_results_df['batWh_80p']
    s_results_df = s_results_df.drop(columns=['batWh_80p'])
except:
    print(f"no batWh_80p column")

safe_env = {"np": np, "nan": np.nan}
for c in strToList:
    s_results_df[c] = s_results_df[c].apply(lambda x: eval(str(x),safe_env))

for c in strToDict:
    s_results_df[c] = s_results_df[c].apply(lambda x: eval(str(x),safe_env))

# Parse only the capex-sensitivity columns
LOADS = ["load1", "load2", "load1R"]
CAPEX_SENS_METRICS = ["NPV", "PBP", "IRR", "LCOSS"]

# gridValue_* stays as-is -- raw input, not derived from capexSensitivity
strToDict = ['gridValue_load1', 'gridValue_load2', 'gridValue_load1R']

capexSensCols = [f"{load}_{metric}_capexSensitivity" for load in LOADS for metric in CAPEX_SENS_METRICS]

safe_env = {"np": np, "nan": np.nan}
for c in strToDict:
    s_results_df[c] = s_results_df[c].apply(lambda x: eval(str(x), safe_env))
for c in capexSensCols:
    s_results_df[c] = s_results_df[c].apply(lambda x: eval(str(x), safe_env))

# Rebuild the standalone {load}_{metric} dict columns from the capexMultiplier=0.0 entry,
# so all existing analysis code (extract_metric, analyze_df, etc.) keeps working unchanged.
for load in LOADS:
    for metric in CAPEX_SENS_METRICS:
        sens_col = f"{load}_{metric}_capexSensitivity"
        standalone_col = f"{load}_{metric}"
        s_results_df[standalone_col] = s_results_df[sens_col].apply(
            lambda d: d.get(0.0, {}) if isinstance(d, dict) else {}
        )

# sanity check -- each rebuilt column should hold a {'without_surplus':.., 'with_surplus':..} dict
print(s_results_df[[f"load1_{m}" for m in CAPEX_SENS_METRICS]].iloc[0])
s_results_df[[f"load1_{m}" for m in CAPEX_SENS_METRICS]].head()

no batWh_80p column
load1_NPV         {'without_surplus': -304.14955314427283, 'with_surplus': 569.141494584254}
load1_PBP         {'without_surplus': 18.861995036290256, 'with_surplus': 4.377518528808113}
load1_IRR      {'without_surplus': -0.11680409044027334, 'with_surplus': 0.2511730816960334}
load1_LCOSS     {'without_surplus': 0.5675630619623425, 'with_surplus': 0.13155400322330182}
Name: 0, dtype: object


,load1_NPV,load1_PBP,load1_IRR,load1_LCOSS
0,"{'without_surplus': -304.14955314427283, 'with_surplus': 569.141494584254}","{'without_surplus': 18.861995036290256, 'with_surplus': 4.377518528808113}","{'without_surplus': -0.11680409044027334, 'with_surplus': 0.2511730816960334}","{'without_surplus': 0.5675630619623425, 'with_surplus': 0.13155400322330182}"
1,"{'without_surplus': -304.14955314427283, 'with_surplus': 569.141494584254}","{'without_surplus': 18.861995036290256, 'with_surplus': 4.377518528808113}","{'without_surplus': -0.11680409044027334, 'with_surplus': 0.2511730816960334}","{'without_surplus': 0.5675630619623425, 'with_surplus': 0.13155400322330182}"
2,"{'without_surplus': -304.14955314427283, 'with_surplus': 569.141494584254}","{'without_surplus': 18.861995036290256, 'with_surplus': 4.377518528808113}","{'without_surplus': -0.11680409044027334, 'with_surplus': 0.2511730816960334}","{'without_surplus': 0.5675630619623425, 'with_surplus': 0.13155400322330182}"
3,"{'without_surplus': -952.423097642707, 'with_surplus': 2373.1399469403427}","{'without_surplus': 25.79965488943166, 'with_surplus': 3.31188833967438}","{'without_surplus': -0.1625210371613503, 'with_surplus': 0.39803314775228493}","{'without_surplus': 0.7229597293362096, 'with_surplus': 0.09254077690261213}"
4,"{'without_surplus': -952.423097642707, 'with_surplus': 2373.1399469403427}","{'without_surplus': 25.79965488943166, 'with_surplus': 3.31188833967438}","{'without_surplus': -0.1625210371613503, 'with_surplus': 0.39803314775228493}","{'without_surplus': 0.7229597293362096, 'with_surplus': 0.09254077690261213}"


In [56]:
s_results_south_df =s_results_df[(s_results_df['az']==180)&(s_results_df['baselineSensitivity']==0)]#&(s_results_df['network']=='bowling green')&(s_results_df['baselineSensitivity']==0)][['capex','pvW','$/Wh']]#.describe()

In [57]:
for l in ['load1','load2','load1R']:
    wS = []
    wSA = []
    for s in s_results_south_df[f'{l}_NPV']:
        wS.append(s['with_surplus'])
        wSA.append(s['with_surplus']) # remove the additional 62.75 once the update is run
    s_results_south_df[f'{l}_NPV_surplus']=wS
    s_results_south_df[f'{l}_NPV_surplus_adjusted']=wSA

    wS = []
    wSA = []
    for s in s_results_south_df[f'{l}_PBP']:
        wS.append(s['with_surplus'])
        wSA.append(s['with_surplus'])
    s_results_south_df[f'{l}_PBP_surplus']=wS
    s_results_south_df[f'{l}_PBP_surplus_adjusted']=wSA

In [58]:
s_results_south_df.describe()

,eventStart,dlrp_rate,csrp_rate,az,pvW,batModel,batWh,batWhAC_80p,effEff,capex,...,load1_PBP_surplus,load1_PBP_surplus_adjusted,load2_NPV_surplus,load2_NPV_surplus_adjusted,load2_PBP_surplus,load2_PBP_surplus_adjusted,load1R_NPV_surplus,load1R_NPV_surplus_adjusted,load1R_PBP_surplus,load1R_PBP_surplus_adjusted
count,16.000000,16.000,16.000000,16.0,16.000000,0.0,16.0,16.0,16.0,16.000000,...,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000
mean,15.625000,20.625,15.000000,180.0,650.000000,NaN,0.0,0.0,0.0,1004.524300,...,3.844703,3.844703,1471.140721,1471.140721,3.844703,3.844703,1471.140721,1471.140721,3.844703,3.844703
std,2.578759,3.500,5.366563,0.0,361.478446,NaN,0.0,0.0,0.0,451.030612,...,0.550289,0.550289,931.580795,931.580795,0.550289,0.550289,931.580795,931.580795,0.550289,0.550289
min,11.000000,18.000,6.000000,180.0,300.000000,NaN,0.0,0.0,0.0,567.815788,...,3.311888,3.311888,569.141495,569.141495,3.311888,3.311888,569.141495,569.141495,3.311888,3.311888
25%,14.000000,18.000,15.000000,180.0,300.000000,NaN,0.0,0.0,0.0,567.815788,...,3.311888,3.311888,569.141495,569.141495,3.311888,3.311888,569.141495,569.141495,3.311888,3.311888
50%,16.000000,18.000,18.000000,180.0,650.000000,NaN,0.0,0.0,0.0,1004.524300,...,3.844703,3.844703,1471.140721,1471.140721,3.844703,3.844703,1471.140721,1471.140721,3.844703,3.844703
75%,16.750000,25.000,18.000000,180.0,1000.000000,NaN,0.0,0.0,0.0,1441.232812,...,4.377519,4.377519,2373.139947,2373.139947,4.377519,4.377519,2373.139947,2373.139947,4.377519,4.377519
max,19.000000,25.000,18.000000,180.0,1000.000000,NaN,0.0,0.0,0.0,1441.232812,...,4.377519,4.377519,2373.139947,2373.139947,4.377519,4.377519,2373.139947,2373.139947,4.377519,4.377519


In [70]:
# Disable row/column truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

s_results_south_df[(s_results_south_df['pvW']==300)&(s_results_south_df['network']=='bowling green')].iloc[0].T

network                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [69]:
s_results_south_df[(s_results_south_df['pvW']==300)&(s_results_south_df['network']=='bowling green')]['gridValue_load2'][0]

{'avg_hourly_surpluskWAC_byMonth': [0.047848551161072894,
  0.0810473729481181,
  0.07108945345847052,
  0.0675996744539619,
  0.08721154963942593,
  0.09092197255950628,
  0.09026620859107835,
  0.0903108817675961,
  0.09861413664319296,
  0.06850659983092788,
  0.06564102836252778,
  0.05655563814177299],
 'med_hourly_surpluskWAC_byMonth': [0.04936178965898763,
  0.07510518419652762,
  0.08438413722882436,
  0.07444567940816332,
  0.07798219251255424,
  0.08350217437927294,
  0.08146179025008214,
  0.0807085259805055,
  0.10920767523146865,
  0.09065390159956162,
  0.06620450150948753,
  0.03942488024446775],
 'min_hourly_surpluskWAC_byMonth': [0.0,
  0.0,
  0.0,
  0.0,
  0.00019985109404015527,
  0.00034968229489387324,
  0.0005473184135709357,
  4.6292541199223054e-05,
  9.450922130385801e-05,
  0.0,
  0.0,
  0.0],
 'max_hourly_surpluskWAC_byMonth': [0.16564720876965977,
  0.21142017051589435,
  0.1880889923648361,
  0.17568648651418012,
  0.2554088870493449,
  0.23824859756141684,